The entire pipeline (source DB → ETL → staging → transformation → core layer with dims and facts) is called Data Warehousing. 
Data Modeling is the sub-discipline that governs how you design the fact and dimension tables in that core layer. You need both.
The reason a separate database exists first is because real businesses already have operational systems running (an e-commerce app, an ERP, a CRM). The data warehouse is built on top of those — it's never a replacement. It reads from them, restructures the data for analytics, and serves BI tools, dashboards, and reports without ever touching the live systems.

Incremental Loading


In [0]:
%sql
CREATE DATABASE sales;

In [0]:
%sql
CREATE TABLE sales.Orders (
  order_id INT,
  order_date DATE,
  customer_id INT,
  customer_name STRING,
  customer_email STRING,
  product_id INT,
  product_name STRING,
  product_category STRING,
  region_id INT,
  region_name STRING,
  country STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  total_amount DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO sales.Orders (order_id, order_date, customer_id, customer_name, customer_email, product_id, product_name, product_category, region_id, region_name, country, quantity, unit_price, total_amount) 
VALUES
(1, '2023-01-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(2, '2023-01-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(3, '2023-01-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(4, '2023-01-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(5, '2023-01-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(6, '2023-01-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(7, '2023-01-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(8, '2023-01-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(9, '2023-01-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(10, '2023-01-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(11, '2023-01-20', 111, 'Ian King', 'iking@example.com', 504, 'Laptop Pro 15', 'Electronics', 2, 'Europe', 'UK', 2, 1500.00, 3000.00),
(12, '2023-01-21', 112, 'Julia Lee', 'jlee@example.com', 502, 'Mechanical Keyboard', 'Accessories', 3, 'Asia', 'India', 1, 85.50, 85.50),
(13, '2023-01-22', 113, 'Kevin Martin', 'kmartin@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 1, 300.00, 300.00),
(14, '2023-01-23', 114, 'Laura Nelson', 'lnelson@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Italy', 4, 12.00, 48.00),
(15, '2023-01-24', 115, 'Mike Owens', 'mowens@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 2, 60.00, 120.00),
(16, '2023-01-25', 101, 'John Doe', 'john.doe@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 2, 999.00, 1998.00),
(17, '2023-01-26', 116, 'Nina Perez', 'nperez@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'Japan', 1, 150.00, 150.00),
(18, '2023-01-27', 117, 'Oscar Quinn', 'oquinn@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'Spain', 3, 25.00, 75.00),
(19, '2023-01-28', 118, 'Paula Roberts', 'proberts@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 2, 15.00, 30.00),
(20, '2023-01-29', 119, 'Quincy Scott', 'qscott@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(21, '2023-01-30', 120, 'Rachel Taylor', 'rtaylor@example.com', 510, 'Webcam 1080p', 'Electronics', 2, 'Europe', 'Germany', 1, 60.00, 60.00),
(22, '2023-01-31', 121, 'Sam Underwood', 'sunderwood@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'South Korea', 1, 1500.00, 1500.00),
(23, '2023-02-01', 122, 'Tina Vance', 'tvance@example.com', 502, 'Mechanical Keyboard', 'Accessories', 1, 'North America', 'USA', 2, 85.50, 171.00),
(24, '2023-02-02', 123, 'Uma Watson', 'uwatson@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'UK', 1, 300.00, 300.00),
(25, '2023-02-03', 124, 'Victor Young', 'vyoung@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'Canada', 10, 12.00, 120.00),
(26, '2023-02-04', 125, 'Wendy Zane', 'wzane@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'India', 1, 999.00, 999.00),
(27, '2023-02-05', 102, 'Jane Smith', 'jane.smith@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'UK', 2, 150.00, 300.00),
(28, '2023-02-06', 126, 'Xavier Adams', 'xadams@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 1, 25.00, 25.00),
(29, '2023-02-07', 127, 'Yvonne Baker', 'ybaker@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'France', 5, 15.00, 75.00),
(30, '2023-02-08', 128, 'Zack Carter', 'zcarter@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 2, 250.00, 500.00),
(31, '2023-02-09', 103, 'Alice Johnson', 'alice.j@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 1, 60.00, 60.00),
(32, '2023-02-10', 129, 'Brian Dixon', 'bdixon@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 3, 1500.00, 4500.00),
(33, '2023-02-11', 130, 'Chloe Edwards', 'cedwards@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'Germany', 2, 85.50, 171.00),
(34, '2023-02-12', 131, 'Daniel Ford', 'dford@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 2, 300.00, 600.00),
(35, '2023-02-13', 132, 'Emma Gomez', 'egomez@example.com', 505, 'USB-C Cable', 'Accessories', 3, 'Asia', 'South Korea', 3, 12.00, 36.00),
(36, '2023-02-14', 133, 'Frank Hughes', 'fhughes@example.com', 506, 'Smartphone X', 'Electronics', 2, 'Europe', 'Spain', 1, 999.00, 999.00),
(37, '2023-02-15', 134, 'Grace Irwin', 'girwin@example.com', 507, 'Bluetooth Headphones', 'Electronics', 1, 'North America', 'Canada', 1, 150.00, 150.00),
(38, '2023-02-16', 104, 'Bob Brown', 'bbrown@example.com', 501, 'Wireless Mouse', 'Accessories', 3, 'Asia', 'Japan', 4, 25.00, 100.00),
(39, '2023-02-17', 135, 'Henry Jones', 'hjones@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'Italy', 2, 15.00, 30.00),
(40, '2023-02-18', 136, 'Ivy Kelly', 'ikelly@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 1, 250.00, 250.00),
(41, '2023-02-19', 137, 'Jack Lewis', 'jlewis@example.com', 510, 'Webcam 1080p', 'Electronics', 3, 'Asia', 'India', 2, 60.00, 120.00),
(42, '2023-02-20', 138, 'Karen Moore', 'kmoore@example.com', 504, 'Laptop Pro 15', 'Electronics', 1, 'North America', 'Canada', 1, 1500.00, 1500.00),
(43, '2023-02-21', 139, 'Leo Nelson', 'lnelson2@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(44, '2023-02-22', 105, 'Charlie Davis', 'cdavis@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'Germany', 3, 300.00, 900.00),
(45, '2023-02-23', 140, 'Mia Ortiz', 'mortiz@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'USA', 6, 12.00, 72.00),
(46, '2023-02-24', 141, 'Noah Perry', 'nperry@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'South Korea', 2, 999.00, 1998.00),
(47, '2023-02-25', 142, 'Olivia Quinn', 'oquinn2@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'France', 1, 150.00, 150.00),
(48, '2023-02-26', 143, 'Paul Reed', 'preed@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'Canada', 2, 25.00, 50.00),
(49, '2023-02-27', 144, 'Quinn Smith', 'qsmith@example.com', 508, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(50, '2023-02-28', 145, 'Ruby Turner', 'rturner@example.com', 509, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00),
(51, '2023-03-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(52, '2023-03-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(53, '2023-03-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(54, '2023-03-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(55, '2023-03-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(56, '2023-03-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(57, '2023-03-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(58, '2023-03-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(59, '2023-03-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(60, '2023-03-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(61, '2023-03-20', 111, 'Ian King', 'iking@example.com', 504, 'Laptop Pro 15', 'Electronics', 2, 'Europe', 'UK', 2, 1500.00, 3000.00),
(62, '2023-03-21', 112, 'Julia Lee', 'jlee@example.com', 502, 'Mechanical Keyboard', 'Accessories', 3, 'Asia', 'India', 1, 85.50, 85.50),
(63, '2023-03-22', 113, 'Kevin Martin', 'kmartin@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 1, 300.00, 300.00),
(64, '2023-03-23', 114, 'Laura Nelson', 'lnelson@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Italy', 4, 12.00, 48.00),
(65, '2023-03-24', 115, 'Mike Owens', 'mowens@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 2, 60.00, 120.00),
(66, '2023-03-25', 101, 'John Doe', 'john.doe@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 2, 999.00, 1998.00),
(67, '2023-03-26', 116, 'Nina Perez', 'nperez@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'Japan', 1, 150.00, 150.00),
(68, '2023-03-27', 117, 'Oscar Quinn', 'oquinn@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'Spain', 3, 25.00, 75.00),
(69, '2023-03-28', 118, 'Paula Roberts', 'proberts@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 2, 15.00, 30.00),
(70, '2023-03-29', 119, 'Quincy Scott', 'qscott@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(71, '2023-03-30', 120, 'Rachel Taylor', 'rtaylor@example.com', 510, 'Webcam 1080p', 'Electronics', 2, 'Europe', 'Germany', 1, 60.00, 60.00),
(72, '2023-03-31', 121, 'Sam Underwood', 'sunderwood@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'South Korea', 1, 1500.00, 1500.00),
(73, '2023-04-01', 122, 'Tina Vance', 'tvance@example.com', 502, 'Mechanical Keyboard', 'Accessories', 1, 'North America', 'USA', 2, 85.50, 171.00),
(74, '2023-04-02', 123, 'Uma Watson', 'uwatson@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'UK', 1, 300.00, 300.00),
(75, '2023-04-03', 124, 'Victor Young', 'vyoung@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'Canada', 10, 12.00, 120.00),
(76, '2023-04-04', 125, 'Wendy Zane', 'wzane@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'India', 1, 999.00, 999.00),
(77, '2023-04-05', 102, 'Jane Smith', 'jane.smith@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'UK', 2, 150.00, 300.00),
(78, '2023-04-06', 126, 'Xavier Adams', 'xadams@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 1, 25.00, 25.00),
(79, '2023-04-07', 127, 'Yvonne Baker', 'ybaker@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'France', 5, 15.00, 75.00),
(80, '2023-04-08', 128, 'Zack Carter', 'zcarter@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 2, 250.00, 500.00),
(81, '2023-04-09', 103, 'Alice Johnson', 'alice.j@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 1, 60.00, 60.00),
(82, '2023-04-10', 129, 'Brian Dixon', 'bdixon@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 3, 1500.00, 4500.00),
(83, '2023-04-11', 130, 'Chloe Edwards', 'cedwards@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'Germany', 2, 85.50, 171.00),
(84, '2023-04-12', 131, 'Daniel Ford', 'dford@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 2, 300.00, 600.00),
(85, '2023-04-13', 132, 'Emma Gomez', 'egomez@example.com', 505, 'USB-C Cable', 'Accessories', 3, 'Asia', 'South Korea', 3, 12.00, 36.00),
(86, '2023-04-14', 133, 'Frank Hughes', 'fhughes@example.com', 506, 'Smartphone X', 'Electronics', 2, 'Europe', 'Spain', 1, 999.00, 999.00),
(87, '2023-04-15', 134, 'Grace Irwin', 'girwin@example.com', 507, 'Bluetooth Headphones', 'Electronics', 1, 'North America', 'Canada', 1, 150.00, 150.00),
(88, '2023-04-16', 104, 'Bob Brown', 'bbrown@example.com', 501, 'Wireless Mouse', 'Accessories', 3, 'Asia', 'Japan', 4, 25.00, 100.00),
(89, '2023-04-17', 135, 'Henry Jones', 'hjones@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'Italy', 2, 15.00, 30.00),
(90, '2023-04-18', 136, 'Ivy Kelly', 'ikelly@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 1, 250.00, 250.00),
(91, '2023-04-19', 137, 'Jack Lewis', 'jlewis@example.com', 510, 'Webcam 1080p', 'Electronics', 3, 'Asia', 'India', 2, 60.00, 120.00),
(92, '2023-04-20', 138, 'Karen Moore', 'kmoore@example.com', 504, 'Laptop Pro 15', 'Electronics', 1, 'North America', 'Canada', 1, 1500.00, 1500.00),
(93, '2023-04-21', 139, 'Leo Nelson', 'lnelson2@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(94, '2023-04-22', 105, 'Charlie Davis', 'cdavis@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'Germany', 3, 300.00, 900.00),
(95, '2023-04-23', 140, 'Mia Ortiz', 'mortiz@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'USA', 6, 12.00, 72.00),
(96, '2023-04-24', 141, 'Noah Perry', 'nperry@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'South Korea', 2, 999.00, 1998.00),
(97, '2023-04-25', 142, 'Olivia Quinn', 'oquinn2@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'France', 1, 150.00, 150.00),
(98, '2023-04-26', 143, 'Paul Reed', 'preed@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'Canada', 2, 25.00, 50.00),
(99, '2023-04-27', 144, 'Quinn Smith', 'qsmith@example.com', 508, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(100, '2023-04-28', 145, 'Ruby Turner', 'rturner@example.com', 509, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00),
(101, '2023-05-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(102, '2023-05-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(103, '2023-05-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(104, '2023-05-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(105, '2023-05-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(106, '2023-05-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(107, '2023-05-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(108, '2023-05-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(109, '2023-05-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(110, '2023-05-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(111, '2023-05-20', 111, 'Ian King', 'iking@example.com', 504, 'Laptop Pro 15', 'Electronics', 2, 'Europe', 'UK', 2, 1500.00, 3000.00),
(112, '2023-05-21', 112, 'Julia Lee', 'jlee@example.com', 502, 'Mechanical Keyboard', 'Accessories', 3, 'Asia', 'India', 1, 85.50, 85.50),
(113, '2023-05-22', 113, 'Kevin Martin', 'kmartin@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 1, 300.00, 300.00),
(114, '2023-05-23', 114, 'Laura Nelson', 'lnelson@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Italy', 4, 12.00, 48.00),
(115, '2023-05-24', 115, 'Mike Owens', 'mowens@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 2, 60.00, 120.00),
(116, '2023-05-25', 101, 'John Doe', 'john.doe@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 2, 999.00, 1998.00),
(117, '2023-05-26', 116, 'Nina Perez', 'nperez@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'Japan', 1, 150.00, 150.00),
(118, '2023-05-27', 117, 'Oscar Quinn', 'oquinn@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'Spain', 3, 25.00, 75.00),
(119, '2023-05-28', 118, 'Paula Roberts', 'proberts@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 2, 15.00, 30.00),
(120, '2023-05-29', 119, 'Quincy Scott', 'qscott@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(121, '2023-05-30', 120, 'Rachel Taylor', 'rtaylor@example.com', 510, 'Webcam 1080p', 'Electronics', 2, 'Europe', 'Germany', 1, 60.00, 60.00),
(122, '2023-05-31', 121, 'Sam Underwood', 'sunderwood@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'South Korea', 1, 1500.00, 1500.00),
(123, '2023-06-01', 122, 'Tina Vance', 'tvance@example.com', 502, 'Mechanical Keyboard', 'Accessories', 1, 'North America', 'USA', 2, 85.50, 171.00),
(124, '2023-06-02', 123, 'Uma Watson', 'uwatson@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'UK', 1, 300.00, 300.00),
(125, '2023-06-03', 124, 'Victor Young', 'vyoung@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'Canada', 10, 12.00, 120.00),
(126, '2023-06-04', 125, 'Wendy Zane', 'wzane@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'India', 1, 999.00, 999.00),
(127, '2023-06-05', 102, 'Jane Smith', 'jane.smith@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'UK', 2, 150.00, 300.00),
(128, '2023-06-06', 126, 'Xavier Adams', 'xadams@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 1, 25.00, 25.00),
(129, '2023-06-07', 127, 'Yvonne Baker', 'ybaker@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'France', 5, 15.00, 75.00),
(130, '2023-06-08', 128, 'Zack Carter', 'zcarter@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 2, 250.00, 500.00),
(131, '2023-06-09', 103, 'Alice Johnson', 'alice.j@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 1, 60.00, 60.00),
(132, '2023-06-10', 129, 'Brian Dixon', 'bdixon@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 3, 1500.00, 4500.00),
(133, '2023-06-11', 130, 'Chloe Edwards', 'cedwards@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'Germany', 2, 85.50, 171.00),
(134, '2023-06-12', 131, 'Daniel Ford', 'dford@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 2, 300.00, 600.00),
(135, '2023-06-13', 132, 'Emma Gomez', 'egomez@example.com', 505, 'USB-C Cable', 'Accessories', 3, 'Asia', 'South Korea', 3, 12.00, 36.00),
(136, '2023-06-14', 133, 'Frank Hughes', 'fhughes@example.com', 506, 'Smartphone X', 'Electronics', 2, 'Europe', 'Spain', 1, 999.00, 999.00),
(137, '2023-06-15', 134, 'Grace Irwin', 'girwin@example.com', 507, 'Bluetooth Headphones', 'Electronics', 1, 'North America', 'Canada', 1, 150.00, 150.00),
(138, '2023-06-16', 104, 'Bob Brown', 'bbrown@example.com', 501, 'Wireless Mouse', 'Accessories', 3, 'Asia', 'Japan', 4, 25.00, 100.00),
(139, '2023-06-17', 135, 'Henry Jones', 'hjones@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'Italy', 2, 15.00, 30.00),
(140, '2023-06-18', 136, 'Ivy Kelly', 'ikelly@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 1, 250.00, 250.00),
(141, '2023-06-19', 137, 'Jack Lewis', 'jlewis@example.com', 510, 'Webcam 1080p', 'Electronics', 3, 'Asia', 'India', 2, 60.00, 120.00),
(142, '2023-06-20', 138, 'Karen Moore', 'kmoore@example.com', 504, 'Laptop Pro 15', 'Electronics', 1, 'North America', 'Canada', 1, 1500.00, 1500.00),
(143, '2023-06-21', 139, 'Leo Nelson', 'lnelson2@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(144, '2023-06-22', 105, 'Charlie Davis', 'cdavis@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'Germany', 3, 300.00, 900.00),
(145, '2023-06-23', 140, 'Mia Ortiz', 'mortiz@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'USA', 6, 12.00, 72.00),
(146, '2023-06-24', 141, 'Noah Perry', 'nperry@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'South Korea', 2, 999.00, 1998.00),
(147, '2023-06-25', 142, 'Olivia Quinn', 'oquinn2@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'France', 1, 150.00, 150.00),
(148, '2023-06-26', 143, 'Paul Reed', 'preed@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'Canada', 2, 25.00, 50.00),
(149, '2023-06-27', 144, 'Quinn Smith', 'qsmith@example.com', 508, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(150, '2023-06-28', 145, 'Ruby Turner', 'rturner@example.com', 509, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00);

num_affected_rows,num_inserted_rows
150,150


Data warehousing

In [0]:
%sql
CREATE DATABASE salesdwh

Staging Layer

In [0]:
%sql
-- inital load
CREATE TABLE salesdwh.staging_sales
AS
SELECT * FROM sales.orders

num_affected_rows,num_inserted_rows


Transformation

In [0]:
%sql
-- multiply qunatity multipled by 10 and not null (Transformation)
CREATE VIEW salesdwh.trans_sales
AS
SELECT *,
    quantity * 10 AS quantity_new
    FROM salesdwh.staging_sales 
    WHERE quantity IS NOT NULL 
    


CORE LAYER

In [0]:
 CREATE TABLE salesdwh.core_sales
 AS
 SELECT * FROM salesdwh.trans_sales

num_affected_rows,num_inserted_rows


Display Corelayer DWH
-- Transformed data is shown in core layer(Dosen't have Data modeling)

In [0]:
SELECT * FROM salesdwh.core_sales

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,region_id,region_name,country,quantity,unit_price,total_amount,quantity_new
1,2023-01-10,101,John Doe,john.doe@example.com,501,Wireless Mouse,Accessories,1,North America,USA,2,25.00,50.00,20
2,2023-01-11,102,Jane Smith,jane.smith@example.com,502,Mechanical Keyboard,Accessories,2,Europe,UK,1,85.50,85.50,10
3,2023-01-12,103,Alice Johnson,alice.j@example.com,503,Gaming Monitor,Electronics,1,North America,Canada,2,300.00,600.00,20
4,2023-01-13,104,Bob Brown,bbrown@example.com,504,Laptop Pro 15,Electronics,3,Asia,Japan,1,1500.00,1500.00,10
5,2023-01-14,105,Charlie Davis,cdavis@example.com,505,USB-C Cable,Accessories,2,Europe,Germany,5,12.00,60.00,50
6,2023-01-15,106,Diana Evans,devans@example.com,506,Smartphone X,Electronics,1,North America,USA,1,999.00,999.00,10
7,2023-01-16,107,Evan Foster,efoster@example.com,507,Bluetooth Headphones,Electronics,3,Asia,South Korea,2,150.00,300.00,20
8,2023-01-17,108,Fiona Green,fgreen@example.com,501,Wireless Mouse,Accessories,2,Europe,France,1,25.00,25.00,10
9,2023-01-18,109,George Harris,gharris@example.com,508,Tablet Screen Protector,Accessories,1,North America,USA,3,15.00,45.00,30
10,2023-01-19,110,Hannah White,hwhite@example.com,509,Smartwatch,Electronics,1,North America,Canada,1,250.00,250.00,10


In [0]:
%sql
-- inserting next 5 days new records 
INSERT INTO sales.Orders (order_id, order_date, customer_id, customer_name, customer_email, product_id, product_name, product_category, region_id, region_name, country, quantity, unit_price, total_amount) 
VALUES
(151, '2023-06-29', 146, 'Quinn Smith', 'qsmith@example.com', 510, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(152, '2023-06-30', 147, 'Ruby Turner', 'rturner@example.com', 511, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00),
(153, '2023-07-01', 148, 'Ethan White', 'ewhite@example.com', 512, 'Headphones', 'Electronics', 1, 'North America', 'USA', 2, 80.00, 160.00),
(154, '2023-07-02', 149, 'Avery Hall', 'ahall@example.com', 513, 'Keyboard', 'Accessories', 4, 'South America', 'Brazil', 1, 40.00, 40.00),
(155, '2023-07-03', 150, 'Mia Jackson', 'mjackson@example.com', 514, 'Laptop Bag', 'Accessories', 5, 'Africa', 'Nigeria', 1, 20.00, 20.00);

num_affected_rows,num_inserted_rows
5,5


In [0]:
SELECT * 
FROM sales.orders
-- 5 new records update (151 to 155) 155 rows


order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,region_id,region_name,country,quantity,unit_price,total_amount
1,2023-01-10,101,John Doe,john.doe@example.com,501,Wireless Mouse,Accessories,1,North America,USA,2,25.00,50.00
2,2023-01-11,102,Jane Smith,jane.smith@example.com,502,Mechanical Keyboard,Accessories,2,Europe,UK,1,85.50,85.50
3,2023-01-12,103,Alice Johnson,alice.j@example.com,503,Gaming Monitor,Electronics,1,North America,Canada,2,300.00,600.00
4,2023-01-13,104,Bob Brown,bbrown@example.com,504,Laptop Pro 15,Electronics,3,Asia,Japan,1,1500.00,1500.00
5,2023-01-14,105,Charlie Davis,cdavis@example.com,505,USB-C Cable,Accessories,2,Europe,Germany,5,12.00,60.00
6,2023-01-15,106,Diana Evans,devans@example.com,506,Smartphone X,Electronics,1,North America,USA,1,999.00,999.00
7,2023-01-16,107,Evan Foster,efoster@example.com,507,Bluetooth Headphones,Electronics,3,Asia,South Korea,2,150.00,300.00
8,2023-01-17,108,Fiona Green,fgreen@example.com,501,Wireless Mouse,Accessories,2,Europe,France,1,25.00,25.00
9,2023-01-18,109,George Harris,gharris@example.com,508,Tablet Screen Protector,Accessories,1,North America,USA,3,15.00,45.00
10,2023-01-19,110,Hannah White,hwhite@example.com,509,Smartwatch,Electronics,1,North America,Canada,1,250.00,250.00


Staging layer Update [Transient] ( For new records and truncate the already available records)

In [0]:
CREATE OR REPLACE TABLE salesDWH.staging_sales
AS
SELECT * FROM sales.orders
WHERE order_date > '2023-06-28' 

num_affected_rows,num_inserted_rows


Transformation (remains same)

In [0]:
%sql
-- multiply qunatity multipled by 10 and not null (Transformation)
-- CREATE VIEW salesdwh.trans_sales
-- AS
-- SELECT *,
--     quantity * 10 AS quantity_new
--     FROM salesdwh.staging_sales 
--     WHERE quantity IS NOT NULL 
    


In [0]:
-- to view last 5 records only
SELECT * FROM salesdwh.trans_sales


order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,region_id,region_name,country,quantity,unit_price,total_amount,quantity_new
151,2023-06-29,146,Quinn Smith,qsmith@example.com,510,Tablet Screen Protector,Accessories,3,Asia,Japan,1,15.00,15.00,10
152,2023-06-30,147,Ruby Turner,rturner@example.com,511,Smartwatch,Electronics,2,Europe,Spain,1,250.00,250.00,10
153,2023-07-01,148,Ethan White,ewhite@example.com,512,Headphones,Electronics,1,North America,USA,2,80.00,160.00,20
154,2023-07-02,149,Avery Hall,ahall@example.com,513,Keyboard,Accessories,4,South America,Brazil,1,40.00,40.00,10
155,2023-07-03,150,Mia Jackson,mjackson@example.com,514,Laptop Bag,Accessories,5,Africa,Nigeria,1,20.00,20.00,10


CORE LAYER (Update)

In [0]:
CREATE OR REPLACE TABLE salesdwh.core_sales (
  order_id INT,
  order_date DATE,
  customer_id INT,
  customer_name STRING,
  customer_email STRING,
  product_id INT,
  product_name STRING,
  product_category STRING,
  region_id INT,
  region_name STRING,
  country STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  total_amount DECIMAL(10,2)
);

In [0]:
-- require data modeling
-- INSERT INTO salesdwh.core_sales
-- SELECT * FROM salesdwh.trans_sales;

Starting from beginging to implement Data modeling. changing the database name to sales_new, (salesdwh to ordersdwh)

In [0]:
CREATE DATABASE sales_new;

In [0]:
%sql
CREATE TABLE sales_new.Orders (
  order_id INT,
  order_date DATE,
  customer_id INT,
  customer_name STRING,
  customer_email STRING,
  product_id INT,
  product_name STRING,
  product_category STRING,
  region_id INT,
  region_name STRING,
  country STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  total_amount DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO sales_new.Orders (order_id, order_date, customer_id, customer_name, customer_email, product_id, product_name, product_category, region_id, region_name, country, quantity, unit_price, total_amount) 
VALUES
(1, '2023-01-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(2, '2023-01-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(3, '2023-01-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(4, '2023-01-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(5, '2023-01-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(6, '2023-01-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(7, '2023-01-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(8, '2023-01-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(9, '2023-01-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(10, '2023-01-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(11, '2023-01-20', 111, 'Ian King', 'iking@example.com', 504, 'Laptop Pro 15', 'Electronics', 2, 'Europe', 'UK', 2, 1500.00, 3000.00),
(12, '2023-01-21', 112, 'Julia Lee', 'jlee@example.com', 502, 'Mechanical Keyboard', 'Accessories', 3, 'Asia', 'India', 1, 85.50, 85.50),
(13, '2023-01-22', 113, 'Kevin Martin', 'kmartin@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 1, 300.00, 300.00),
(14, '2023-01-23', 114, 'Laura Nelson', 'lnelson@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Italy', 4, 12.00, 48.00),
(15, '2023-01-24', 115, 'Mike Owens', 'mowens@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 2, 60.00, 120.00),
(16, '2023-01-25', 101, 'John Doe', 'john.doe@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 2, 999.00, 1998.00),
(17, '2023-01-26', 116, 'Nina Perez', 'nperez@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'Japan', 1, 150.00, 150.00),
(18, '2023-01-27', 117, 'Oscar Quinn', 'oquinn@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'Spain', 3, 25.00, 75.00),
(19, '2023-01-28', 118, 'Paula Roberts', 'proberts@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 2, 15.00, 30.00),
(20, '2023-01-29', 119, 'Quincy Scott', 'qscott@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(21, '2023-01-30', 120, 'Rachel Taylor', 'rtaylor@example.com', 510, 'Webcam 1080p', 'Electronics', 2, 'Europe', 'Germany', 1, 60.00, 60.00),
(22, '2023-01-31', 121, 'Sam Underwood', 'sunderwood@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'South Korea', 1, 1500.00, 1500.00),
(23, '2023-02-01', 122, 'Tina Vance', 'tvance@example.com', 502, 'Mechanical Keyboard', 'Accessories', 1, 'North America', 'USA', 2, 85.50, 171.00),
(24, '2023-02-02', 123, 'Uma Watson', 'uwatson@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'UK', 1, 300.00, 300.00),
(25, '2023-02-03', 124, 'Victor Young', 'vyoung@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'Canada', 10, 12.00, 120.00),
(26, '2023-02-04', 125, 'Wendy Zane', 'wzane@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'India', 1, 999.00, 999.00),
(27, '2023-02-05', 102, 'Jane Smith', 'jane.smith@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'UK', 2, 150.00, 300.00),
(28, '2023-02-06', 126, 'Xavier Adams', 'xadams@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 1, 25.00, 25.00),
(29, '2023-02-07', 127, 'Yvonne Baker', 'ybaker@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'France', 5, 15.00, 75.00),
(30, '2023-02-08', 128, 'Zack Carter', 'zcarter@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 2, 250.00, 500.00),
(31, '2023-02-09', 103, 'Alice Johnson', 'alice.j@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 1, 60.00, 60.00),
(32, '2023-02-10', 129, 'Brian Dixon', 'bdixon@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 3, 1500.00, 4500.00),
(33, '2023-02-11', 130, 'Chloe Edwards', 'cedwards@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'Germany', 2, 85.50, 171.00),
(34, '2023-02-12', 131, 'Daniel Ford', 'dford@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 2, 300.00, 600.00),
(35, '2023-02-13', 132, 'Emma Gomez', 'egomez@example.com', 505, 'USB-C Cable', 'Accessories', 3, 'Asia', 'South Korea', 3, 12.00, 36.00),
(36, '2023-02-14', 133, 'Frank Hughes', 'fhughes@example.com', 506, 'Smartphone X', 'Electronics', 2, 'Europe', 'Spain', 1, 999.00, 999.00),
(37, '2023-02-15', 134, 'Grace Irwin', 'girwin@example.com', 507, 'Bluetooth Headphones', 'Electronics', 1, 'North America', 'Canada', 1, 150.00, 150.00),
(38, '2023-02-16', 104, 'Bob Brown', 'bbrown@example.com', 501, 'Wireless Mouse', 'Accessories', 3, 'Asia', 'Japan', 4, 25.00, 100.00),
(39, '2023-02-17', 135, 'Henry Jones', 'hjones@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'Italy', 2, 15.00, 30.00),
(40, '2023-02-18', 136, 'Ivy Kelly', 'ikelly@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 1, 250.00, 250.00),
(41, '2023-02-19', 137, 'Jack Lewis', 'jlewis@example.com', 510, 'Webcam 1080p', 'Electronics', 3, 'Asia', 'India', 2, 60.00, 120.00),
(42, '2023-02-20', 138, 'Karen Moore', 'kmoore@example.com', 504, 'Laptop Pro 15', 'Electronics', 1, 'North America', 'Canada', 1, 1500.00, 1500.00),
(43, '2023-02-21', 139, 'Leo Nelson', 'lnelson2@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(44, '2023-02-22', 105, 'Charlie Davis', 'cdavis@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'Germany', 3, 300.00, 900.00),
(45, '2023-02-23', 140, 'Mia Ortiz', 'mortiz@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'USA', 6, 12.00, 72.00),
(46, '2023-02-24', 141, 'Noah Perry', 'nperry@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'South Korea', 2, 999.00, 1998.00),
(47, '2023-02-25', 142, 'Olivia Quinn', 'oquinn2@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'France', 1, 150.00, 150.00),
(48, '2023-02-26', 143, 'Paul Reed', 'preed@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'Canada', 2, 25.00, 50.00),
(49, '2023-02-27', 144, 'Quinn Smith', 'qsmith@example.com', 508, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(50, '2023-02-28', 145, 'Ruby Turner', 'rturner@example.com', 509, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00),
(51, '2023-03-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(52, '2023-03-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(53, '2023-03-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(54, '2023-03-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(55, '2023-03-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(56, '2023-03-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(57, '2023-03-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(58, '2023-03-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(59, '2023-03-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(60, '2023-03-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(61, '2023-03-20', 111, 'Ian King', 'iking@example.com', 504, 'Laptop Pro 15', 'Electronics', 2, 'Europe', 'UK', 2, 1500.00, 3000.00),
(62, '2023-03-21', 112, 'Julia Lee', 'jlee@example.com', 502, 'Mechanical Keyboard', 'Accessories', 3, 'Asia', 'India', 1, 85.50, 85.50),
(63, '2023-03-22', 113, 'Kevin Martin', 'kmartin@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 1, 300.00, 300.00),
(64, '2023-03-23', 114, 'Laura Nelson', 'lnelson@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Italy', 4, 12.00, 48.00),
(65, '2023-03-24', 115, 'Mike Owens', 'mowens@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 2, 60.00, 120.00),
(66, '2023-03-25', 101, 'John Doe', 'john.doe@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 2, 999.00, 1998.00),
(67, '2023-03-26', 116, 'Nina Perez', 'nperez@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'Japan', 1, 150.00, 150.00),
(68, '2023-03-27', 117, 'Oscar Quinn', 'oquinn@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'Spain', 3, 25.00, 75.00),
(69, '2023-03-28', 118, 'Paula Roberts', 'proberts@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 2, 15.00, 30.00),
(70, '2023-03-29', 119, 'Quincy Scott', 'qscott@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(71, '2023-03-30', 120, 'Rachel Taylor', 'rtaylor@example.com', 510, 'Webcam 1080p', 'Electronics', 2, 'Europe', 'Germany', 1, 60.00, 60.00),
(72, '2023-03-31', 121, 'Sam Underwood', 'sunderwood@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'South Korea', 1, 1500.00, 1500.00),
(73, '2023-04-01', 122, 'Tina Vance', 'tvance@example.com', 502, 'Mechanical Keyboard', 'Accessories', 1, 'North America', 'USA', 2, 85.50, 171.00),
(74, '2023-04-02', 123, 'Uma Watson', 'uwatson@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'UK', 1, 300.00, 300.00),
(75, '2023-04-03', 124, 'Victor Young', 'vyoung@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'Canada', 10, 12.00, 120.00),
(76, '2023-04-04', 125, 'Wendy Zane', 'wzane@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'India', 1, 999.00, 999.00),
(77, '2023-04-05', 102, 'Jane Smith', 'jane.smith@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'UK', 2, 150.00, 300.00),
(78, '2023-04-06', 126, 'Xavier Adams', 'xadams@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 1, 25.00, 25.00),
(79, '2023-04-07', 127, 'Yvonne Baker', 'ybaker@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'France', 5, 15.00, 75.00),
(80, '2023-04-08', 128, 'Zack Carter', 'zcarter@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 2, 250.00, 500.00),
(81, '2023-04-09', 103, 'Alice Johnson', 'alice.j@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 1, 60.00, 60.00),
(82, '2023-04-10', 129, 'Brian Dixon', 'bdixon@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 3, 1500.00, 4500.00),
(83, '2023-04-11', 130, 'Chloe Edwards', 'cedwards@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'Germany', 2, 85.50, 171.00),
(84, '2023-04-12', 131, 'Daniel Ford', 'dford@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 2, 300.00, 600.00),
(85, '2023-04-13', 132, 'Emma Gomez', 'egomez@example.com', 505, 'USB-C Cable', 'Accessories', 3, 'Asia', 'South Korea', 3, 12.00, 36.00),
(86, '2023-04-14', 133, 'Frank Hughes', 'fhughes@example.com', 506, 'Smartphone X', 'Electronics', 2, 'Europe', 'Spain', 1, 999.00, 999.00),
(87, '2023-04-15', 134, 'Grace Irwin', 'girwin@example.com', 507, 'Bluetooth Headphones', 'Electronics', 1, 'North America', 'Canada', 1, 150.00, 150.00),
(88, '2023-04-16', 104, 'Bob Brown', 'bbrown@example.com', 501, 'Wireless Mouse', 'Accessories', 3, 'Asia', 'Japan', 4, 25.00, 100.00),
(89, '2023-04-17', 135, 'Henry Jones', 'hjones@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'Italy', 2, 15.00, 30.00),
(90, '2023-04-18', 136, 'Ivy Kelly', 'ikelly@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 1, 250.00, 250.00),
(91, '2023-04-19', 137, 'Jack Lewis', 'jlewis@example.com', 510, 'Webcam 1080p', 'Electronics', 3, 'Asia', 'India', 2, 60.00, 120.00),
(92, '2023-04-20', 138, 'Karen Moore', 'kmoore@example.com', 504, 'Laptop Pro 15', 'Electronics', 1, 'North America', 'Canada', 1, 1500.00, 1500.00),
(93, '2023-04-21', 139, 'Leo Nelson', 'lnelson2@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(94, '2023-04-22', 105, 'Charlie Davis', 'cdavis@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'Germany', 3, 300.00, 900.00),
(95, '2023-04-23', 140, 'Mia Ortiz', 'mortiz@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'USA', 6, 12.00, 72.00),
(96, '2023-04-24', 141, 'Noah Perry', 'nperry@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'South Korea', 2, 999.00, 1998.00),
(97, '2023-04-25', 142, 'Olivia Quinn', 'oquinn2@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'France', 1, 150.00, 150.00),
(98, '2023-04-26', 143, 'Paul Reed', 'preed@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'Canada', 2, 25.00, 50.00),
(99, '2023-04-27', 144, 'Quinn Smith', 'qsmith@example.com', 508, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(100, '2023-04-28', 145, 'Ruby Turner', 'rturner@example.com', 509, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00),
(101, '2023-05-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(102, '2023-05-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(103, '2023-05-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(104, '2023-05-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(105, '2023-05-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(106, '2023-05-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(107, '2023-05-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(108, '2023-05-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(109, '2023-05-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(110, '2023-05-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(111, '2023-05-20', 111, 'Ian King', 'iking@example.com', 504, 'Laptop Pro 15', 'Electronics', 2, 'Europe', 'UK', 2, 1500.00, 3000.00),
(112, '2023-05-21', 112, 'Julia Lee', 'jlee@example.com', 502, 'Mechanical Keyboard', 'Accessories', 3, 'Asia', 'India', 1, 85.50, 85.50),
(113, '2023-05-22', 113, 'Kevin Martin', 'kmartin@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 1, 300.00, 300.00),
(114, '2023-05-23', 114, 'Laura Nelson', 'lnelson@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Italy', 4, 12.00, 48.00),
(115, '2023-05-24', 115, 'Mike Owens', 'mowens@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 2, 60.00, 120.00),
(116, '2023-05-25', 101, 'John Doe', 'john.doe@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 2, 999.00, 1998.00),
(117, '2023-05-26', 116, 'Nina Perez', 'nperez@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'Japan', 1, 150.00, 150.00),
(118, '2023-05-27', 117, 'Oscar Quinn', 'oquinn@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'Spain', 3, 25.00, 75.00),
(119, '2023-05-28', 118, 'Paula Roberts', 'proberts@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 2, 15.00, 30.00),
(120, '2023-05-29', 119, 'Quincy Scott', 'qscott@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00),
(121, '2023-05-30', 120, 'Rachel Taylor', 'rtaylor@example.com', 510, 'Webcam 1080p', 'Electronics', 2, 'Europe', 'Germany', 1, 60.00, 60.00),
(122, '2023-05-31', 121, 'Sam Underwood', 'sunderwood@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'South Korea', 1, 1500.00, 1500.00),
(123, '2023-06-01', 122, 'Tina Vance', 'tvance@example.com', 502, 'Mechanical Keyboard', 'Accessories', 1, 'North America', 'USA', 2, 85.50, 171.00),
(124, '2023-06-02', 123, 'Uma Watson', 'uwatson@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'UK', 1, 300.00, 300.00),
(125, '2023-06-03', 124, 'Victor Young', 'vyoung@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'Canada', 10, 12.00, 120.00),
(126, '2023-06-04', 125, 'Wendy Zane', 'wzane@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'India', 1, 999.00, 999.00),
(127, '2023-06-05', 102, 'Jane Smith', 'jane.smith@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'UK', 2, 150.00, 300.00),
(128, '2023-06-06', 126, 'Xavier Adams', 'xadams@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 1, 25.00, 25.00),
(129, '2023-06-07', 127, 'Yvonne Baker', 'ybaker@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'France', 5, 15.00, 75.00),
(130, '2023-06-08', 128, 'Zack Carter', 'zcarter@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 2, 250.00, 500.00),
(131, '2023-06-09', 103, 'Alice Johnson', 'alice.j@example.com', 510, 'Webcam 1080p', 'Electronics', 1, 'North America', 'Canada', 1, 60.00, 60.00),
(132, '2023-06-10', 129, 'Brian Dixon', 'bdixon@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 3, 1500.00, 4500.00),
(133, '2023-06-11', 130, 'Chloe Edwards', 'cedwards@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'Germany', 2, 85.50, 171.00),
(134, '2023-06-12', 131, 'Daniel Ford', 'dford@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'USA', 2, 300.00, 600.00),
(135, '2023-06-13', 132, 'Emma Gomez', 'egomez@example.com', 505, 'USB-C Cable', 'Accessories', 3, 'Asia', 'South Korea', 3, 12.00, 36.00),
(136, '2023-06-14', 133, 'Frank Hughes', 'fhughes@example.com', 506, 'Smartphone X', 'Electronics', 2, 'Europe', 'Spain', 1, 999.00, 999.00),
(137, '2023-06-15', 134, 'Grace Irwin', 'girwin@example.com', 507, 'Bluetooth Headphones', 'Electronics', 1, 'North America', 'Canada', 1, 150.00, 150.00),
(138, '2023-06-16', 104, 'Bob Brown', 'bbrown@example.com', 501, 'Wireless Mouse', 'Accessories', 3, 'Asia', 'Japan', 4, 25.00, 100.00),
(139, '2023-06-17', 135, 'Henry Jones', 'hjones@example.com', 508, 'Tablet Screen Protector', 'Accessories', 2, 'Europe', 'Italy', 2, 15.00, 30.00),
(140, '2023-06-18', 136, 'Ivy Kelly', 'ikelly@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'USA', 1, 250.00, 250.00),
(141, '2023-06-19', 137, 'Jack Lewis', 'jlewis@example.com', 510, 'Webcam 1080p', 'Electronics', 3, 'Asia', 'India', 2, 60.00, 120.00),
(142, '2023-06-20', 138, 'Karen Moore', 'kmoore@example.com', 504, 'Laptop Pro 15', 'Electronics', 1, 'North America', 'Canada', 1, 1500.00, 1500.00),
(143, '2023-06-21', 139, 'Leo Nelson', 'lnelson2@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(144, '2023-06-22', 105, 'Charlie Davis', 'cdavis@example.com', 503, 'Gaming Monitor', 'Electronics', 2, 'Europe', 'Germany', 3, 300.00, 900.00),
(145, '2023-06-23', 140, 'Mia Ortiz', 'mortiz@example.com', 505, 'USB-C Cable', 'Accessories', 1, 'North America', 'USA', 6, 12.00, 72.00),
(146, '2023-06-24', 141, 'Noah Perry', 'nperry@example.com', 506, 'Smartphone X', 'Electronics', 3, 'Asia', 'South Korea', 2, 999.00, 1998.00),
(147, '2023-06-25', 142, 'Olivia Quinn', 'oquinn2@example.com', 507, 'Bluetooth Headphones', 'Electronics', 2, 'Europe', 'France', 1, 150.00, 150.00),
(148, '2023-06-26', 143, 'Paul Reed', 'preed@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'Canada', 2, 25.00, 50.00),
(149, '2023-06-27', 144, 'Quinn Smith', 'qsmith@example.com', 508, 'Tablet Screen Protector', 'Accessories', 3, 'Asia', 'Japan', 1, 15.00, 15.00),
(150, '2023-06-28', 145, 'Ruby Turner', 'rturner@example.com', 509, 'Smartwatch', 'Electronics', 2, 'Europe', 'Spain', 1, 250.00, 250.00);

num_affected_rows,num_inserted_rows
150,150


In [0]:
SELECT * FROM sales_new.orders

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,region_id,region_name,country,quantity,unit_price,total_amount
1,2023-01-10,101,John Doe,john.doe@example.com,501,Wireless Mouse,Accessories,1,North America,USA,2,25.00,50.00
2,2023-01-11,102,Jane Smith,jane.smith@example.com,502,Mechanical Keyboard,Accessories,2,Europe,UK,1,85.50,85.50
3,2023-01-12,103,Alice Johnson,alice.j@example.com,503,Gaming Monitor,Electronics,1,North America,Canada,2,300.00,600.00
4,2023-01-13,104,Bob Brown,bbrown@example.com,504,Laptop Pro 15,Electronics,3,Asia,Japan,1,1500.00,1500.00
5,2023-01-14,105,Charlie Davis,cdavis@example.com,505,USB-C Cable,Accessories,2,Europe,Germany,5,12.00,60.00
6,2023-01-15,106,Diana Evans,devans@example.com,506,Smartphone X,Electronics,1,North America,USA,1,999.00,999.00
7,2023-01-16,107,Evan Foster,efoster@example.com,507,Bluetooth Headphones,Electronics,3,Asia,South Korea,2,150.00,300.00
8,2023-01-17,108,Fiona Green,fgreen@example.com,501,Wireless Mouse,Accessories,2,Europe,France,1,25.00,25.00
9,2023-01-18,109,George Harris,gharris@example.com,508,Tablet Screen Protector,Accessories,1,North America,USA,3,15.00,45.00
10,2023-01-19,110,Hannah White,hwhite@example.com,509,Smartwatch,Electronics,1,North America,Canada,1,250.00,250.00


Data warehouse creation

In [0]:
%sql
CREATE DATABASE ordersdwh

Staging Layer

In [0]:
CREATE OR REPLACE TABLE ordersdwh.staging_sales
AS SELECT * FROM sales_new.orders

num_affected_rows,num_inserted_rows


Transformation

In [0]:
CREATE OR REPLACE VIEW ordersdwh.transform_sales
AS
SELECT *,
    quantity * 10 AS quantity_new
    FROM ordersdwh.staging_sales
    WHERE quantity IS NOT NULL 
    

CORE LAYER with Data modelling to share this with other Data Specialist[Analyst, Scientist] (With dim and fact tables) 

In [0]:
SELECT * FROM ordersdwh.transform_sales
-- full data 

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,region_id,region_name,country,quantity,unit_price,total_amount,quantity_new
1,2023-01-10,101,John Doe,john.doe@example.com,501,Wireless Mouse,Accessories,1,North America,USA,2,25.00,50.00,20
2,2023-01-11,102,Jane Smith,jane.smith@example.com,502,Mechanical Keyboard,Accessories,2,Europe,UK,1,85.50,85.50,10
3,2023-01-12,103,Alice Johnson,alice.j@example.com,503,Gaming Monitor,Electronics,1,North America,Canada,2,300.00,600.00,20
4,2023-01-13,104,Bob Brown,bbrown@example.com,504,Laptop Pro 15,Electronics,3,Asia,Japan,1,1500.00,1500.00,10
5,2023-01-14,105,Charlie Davis,cdavis@example.com,505,USB-C Cable,Accessories,2,Europe,Germany,5,12.00,60.00,50
6,2023-01-15,106,Diana Evans,devans@example.com,506,Smartphone X,Electronics,1,North America,USA,1,999.00,999.00,10
7,2023-01-16,107,Evan Foster,efoster@example.com,507,Bluetooth Headphones,Electronics,3,Asia,South Korea,2,150.00,300.00,20
8,2023-01-17,108,Fiona Green,fgreen@example.com,501,Wireless Mouse,Accessories,2,Europe,France,1,25.00,25.00,10
9,2023-01-18,109,George Harris,gharris@example.com,508,Tablet Screen Protector,Accessories,1,North America,USA,3,15.00,45.00,30
10,2023-01-19,110,Hannah White,hwhite@example.com,509,Smartwatch,Electronics,1,North America,Canada,1,250.00,250.00,10


#Breaking the data into DIMENSIONS and FACT TABLES TO SERVE THE core  layer
Create Dimensions first because Fact tables requires surrogate keys from Dimensions tables. 

###Customer Dimension 
[Dimension rules - non aregatable numeric col, should have surrogate key. "Records" Should be Dim[1] relation to [many] fact tabel]
Surrogate key is the connector/ Joining for the Fact table. Dimension can't have duplicate




![image_1773153682893.png](./image_1773153682893.png "image_1773153682893.png")

Dimcustomer

-- SELECT 
  DISTINCT(customer_id) AS customer_id,
  customer_name,
  customer_email
  FROM ordersdwh.transform_sales

  Joiner with Fact table (Surrogate key is missed)

In [0]:
%sql
CREATE OR REPLACE TABLE ordersdwh.Dimcustomers
(
  cust_id INT,
  cust_name VARCHAR(120),
  cust_email VARCHAR(120),
  dimcust_key INT
)

In [0]:
CREATE OR REPLACE VIEW ordersdwh.viewdim_cust
AS
SELECT T.*,
  ROW_NUMBER() OVER(ORDER BY T.customer_id) AS dimcustomer_key
FROM 
(
  SELECT 
    DISTINCT(customer_id) AS customer_id,
    customer_name,
    customer_email
  FROM ordersdwh.transform_sales
)AS T


In [0]:
%sql
SELECT * FROM ordersdwh.viewdim_cust

customer_id,customer_name,customer_email,dimcustomer_key
101,John Doe,john.doe@example.com,1
102,Jane Smith,jane.smith@example.com,2
103,Alice Johnson,alice.j@example.com,3
104,Bob Brown,bbrown@example.com,4
105,Charlie Davis,cdavis@example.com,5
106,Diana Evans,devans@example.com,6
107,Evan Foster,efoster@example.com,7
108,Fiona Green,fgreen@example.com,8
109,George Harris,gharris@example.com,9
110,Hannah White,hwhite@example.com,10


In [0]:
INSERT INTO ordersdwh.dimcustomers
(
  SELECT * FROM ordersdwh.viewdim_cust
)

num_affected_rows,num_inserted_rows
45,45


In [0]:
SELECT * FROM ordersdwh.dimcustomers

cust_id,cust_name,cust_email,dimcust_key
101,John Doe,john.doe@example.com,1
102,Jane Smith,jane.smith@example.com,2
103,Alice Johnson,alice.j@example.com,3
104,Bob Brown,bbrown@example.com,4
105,Charlie Davis,cdavis@example.com,5
106,Diana Evans,devans@example.com,6
107,Evan Foster,efoster@example.com,7
108,Fiona Green,fgreen@example.com,8
109,George Harris,gharris@example.com,9
110,Hannah White,hwhite@example.com,10


Dimension - Dimproducts

In [0]:
%sql
DESCRIBE sales_new.orders

col_name,data_type,comment
order_id,int,null
order_date,date,null
customer_id,int,null
customer_name,string,null
customer_email,string,null
product_id,int,null
product_name,string,null
product_category,string,null
region_id,int,null
region_name,string,null


In [0]:
CREATE OR REPLACE TABLE ordersdwh.dimproducts
(
  product_id INT,
  product_name VARCHAR(120),
  product_category VARCHAR(120),
  dimproduct_key INT
)



In [0]:
CREATE OR REPLACE VIEW ordersdwh.viewdimproducts
AS 
SELECT T.*,
  ROW_NUMBER() OVER(ORDER BY T.product_id) AS dimproduct_key
FROM
(SELECT 
  DISTINCT(product_id) AS product_id,
  product_name,
  product_category
FROM 
ordersdwh.transform_sales
) AS T


In [0]:
INSERT INTO ordersdwh.dimproducts
SELECT * FROM ordersdwh.viewdimproducts

num_affected_rows,num_inserted_rows
10,10


In [0]:
SELECT * FROM ordersdwh.dimproducts

product_id,product_name,product_category,dimproduct_key
501,Wireless Mouse,Accessories,1
502,Mechanical Keyboard,Accessories,2
503,Gaming Monitor,Electronics,3
504,Laptop Pro 15,Electronics,4
505,USB-C Cable,Accessories,5
506,Smartphone X,Electronics,6
507,Bluetooth Headphones,Electronics,7
508,Tablet Screen Protector,Accessories,8
509,Smartwatch,Electronics,9
510,Webcam 1080p,Electronics,10


Dimension - Dimension Region

In [0]:
CREATE OR REPLACE TABLE ordersdwh.dimregion
(
  region_id INT,
  region_name VARCHAR(120),
  country VARCHAR(120),
  dimregion_key INT
)

In [0]:
CREATE OR REPLACE VIEW ordersdwh.viewdim_region
AS
SELECT T.*,
 ROW_NUMBER() OVER(ORDER BY T.region_id) AS dimregion_key
FROM
(SELECT 
 DISTINCT(region_id) AS region_id,
 region_name,
 country
FROM 
ordersdwh.transform_sales
) AS T

In [0]:
INSERT INTO ordersdwh.dimregion
SELECT * FROM ordersdwh.viewdim_region

num_affected_rows,num_inserted_rows
10,10


In [0]:
SELECT * FROM ordersdwh.dimregion

region_id,region_name,country,dimregion_key
1,North America,USA,1
1,North America,Canada,2
2,Europe,Spain,3
2,Europe,France,4
2,Europe,Italy,5
2,Europe,Germany,6
2,Europe,UK,7
3,Asia,South Korea,8
3,Asia,Japan,9
3,Asia,India,10


Dimension Datedim

In [0]:
CREATE OR REPLACE TABLE ordersdwh.dimdate
(
  order_date DATE,
  dimdate_key INT
)

In [0]:
CREATE OR REPLACE VIEW ordersdwh.viewdimdate
AS
SELECT T.*, 
  ROW_NUMBER() OVER(ORDER BY T.order_date) AS dimdate_key
FROM
(SELECT DISTINCT(order_date) AS order_date
FROM ordersdwh.transform_sales)
AS T


In [0]:
INSERT INTO ordersdwh.dimdate
SELECT * FROM ordersdwh.viewdimdate


num_affected_rows,num_inserted_rows
150,150


In [0]:
SELECT * FROM ordersdwh.dimdate

order_date,dimdate_key
2023-01-10,1
2023-01-11,2
2023-01-12,3
2023-01-13,4
2023-01-14,5
2023-01-15,6
2023-01-16,7
2023-01-17,8
2023-01-18,9
2023-01-19,10


Fact Table Creation

In [0]:
CREATE OR REPLACE TABLE ordersdwh.factsales
(
  orderid INT,
  quantity INT,
  unit_price DECIMAL(10,2),
  total_amount DECIMAL(10,2),
  dimcust_key INT,
  dimproduct_key INT,
  dimregion_key INT,
  dimdate_key  INT
)



The result of the below code show the fact of dimcustomer and the orderdwh.transformation sales( check script 38 for reference). The customer id 101 in the order id 16 is replaced with 1(dimcust_key) shows the fact table joined with customer dimension

In [0]:
CREATE OR REPLACE VIEW ordersdwh.viewfactsales
AS
SELECT 
  f.order_id,
  f.quantity,
  f.unit_price,
  f.total_amount,
  dc.dimcust_key,
  dp.dimproduct_key,
  dr.dimregion_key,
  dd.dimdate_key 
FROM 

ordersdwh.transform_sales f
LEFT JOIN ordersdwh.dimcustomers dc
ON f.customer_id = dc.cust_id

LEFT JOIN ordersdwh.dimproducts dp
ON f.product_id = dp.product_id

LEFT JOIN ordersdwh.dimregion dr
ON f.country = dr.country

LEFT JOIN ordersdwh.dimdate dd
ON f.order_date = dd.order_date

In [0]:
INSERT INTO ordersdwh.factsales
SELECT * FROM ordersdwh.viewfactsales

num_affected_rows,num_inserted_rows
150,150


In [0]:
SELECT * FROM ordersdwh.factsales

orderid,quantity,unit_price,total_amount,dimcust_key,dimproduct_key,dimregion_key,dimdate_key
1,2,25.00,50.00,1,1,1,1
2,1,85.50,85.50,2,2,7,2
3,2,300.00,600.00,3,3,2,3
4,1,1500.00,1500.00,4,4,9,4
5,5,12.00,60.00,5,5,6,5
6,1,999.00,999.00,6,6,1,6
7,2,150.00,300.00,7,7,8,7
8,1,25.00,25.00,8,1,4,8
9,3,15.00,45.00,9,8,1,9
10,1,250.00,250.00,10,9,2,10


#SCD

In [0]:
CREATE DATABASE sales_scd;

In [0]:
%sql
CREATE TABLE sales_scd.orders (
  order_id INT,
  order_date DATE,
  customer_id INT,
  customer_name STRING,
  customer_email STRING,
  product_id INT,
  product_name STRING,
  product_category STRING,
  region_id INT,
  region_name STRING,
  country STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  total_amount DECIMAL(10,2)
);

In [0]:
INSERT INTO sales_scd.orders (order_id, order_date, customer_id, customer_name, customer_email, product_id, product_name, product_category, region_id, region_name, country, quantity, unit_price, total_amount) 
VALUES
(1, '2023-01-10', 101, 'John Doe', 'john.doe@example.com', 501, 'Wireless Mouse', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(2, '2023-01-11', 102, 'Jane Smith', 'jane.smith@example.com', 502, 'Mechanical Keyboard', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50),
(3, '2023-01-12', 103, 'Alice Johnson', 'alice.j@example.com', 503, 'Gaming Monitor', 'Electronics', 1, 'North America', 'Canada', 2, 300.00, 600.00),
(4, '2023-01-13', 104, 'Bob Brown', 'bbrown@example.com', 504, 'Laptop Pro 15', 'Electronics', 3, 'Asia', 'Japan', 1, 1500.00, 1500.00),
(5, '2023-01-14', 105, 'Charlie Davis', 'cdavis@example.com', 505, 'USB-C Cable', 'Accessories', 2, 'Europe', 'Germany', 5, 12.00, 60.00),
(6, '2023-01-15', 106, 'Diana Evans', 'devans@example.com', 506, 'Smartphone X', 'Electronics', 1, 'North America', 'USA', 1, 999.00, 999.00),
(7, '2023-01-16', 107, 'Evan Foster', 'efoster@example.com', 507, 'Bluetooth Headphones', 'Electronics', 3, 'Asia', 'South Korea', 2, 150.00, 300.00),
(8, '2023-01-17', 108, 'Fiona Green', 'fgreen@example.com', 501, 'Wireless Mouse', 'Accessories', 2, 'Europe', 'France', 1, 25.00, 25.00),
(9, '2023-01-18', 109, 'George Harris', 'gharris@example.com', 508, 'Tablet Screen Protector', 'Accessories', 1, 'North America', 'USA', 3, 15.00, 45.00),
(10, '2023-01-19', 110, 'Hannah White', 'hwhite@example.com', 509, 'Smartwatch', 'Electronics', 1, 'North America', 'Canada', 1, 250.00, 250.00)


num_affected_rows,num_inserted_rows
10,10


SCD TYPE 1 UPSERRT (UPDATE+INSERT) new upcoming data

In [0]:
CREATE OR REPLACE VIEW sales_scd.viewscdproducts
AS
(
SELECT DISTINCT(product_id) as product_id, product_name,product_category FROM sales_scd.orders
)

In [0]:
CREATE OR REPLACE TABLE sales_scd.dimproducts
(
  product_id INT,
  product_name VARCHAR(120),
  product_category VARCHAR(120)
)

In [0]:
INSERT INTO sales_scd.dimproducts
SELECT * FROM sales_scd.viewscdproducts

num_affected_rows,num_inserted_rows
9,9


In [0]:
SELECT * FROM sales_scd.dimproducts

product_id,product_name,product_category
508,Tablet Screen Protector,Accessories
502,Mechanical Keyboard,Accessories
504,Laptop Pro 15,Electronics
507,Bluetooth Headphones,Electronics
503,Gaming Monitor,Electronics
509,Smartwatch,Electronics
506,Smartphone X,Electronics
501,Wireless Mouse,Accessories
505,USB-C Cable,Accessories


In [0]:
INSERT INTO sales_scd.orders (order_id, order_date, customer_id, customer_name, customer_email, product_id, product_name, product_category, region_id, region_name, country, quantity, unit_price, total_amount) 
VALUES
(1, '2023-01-20', 101, 'John Doe', 'john.doe@example.com', 501, 'Airpods', 'Accessories', 1, 'North America', 'USA', 2, 25.00, 50.00),
(2, '2023-01-21', 102, 'Jane Smith', 'jane.smith@example.com', 504, 'Gaming laptop', 'Accessories', 2, 'Europe', 'UK', 1, 85.50, 85.50)

num_affected_rows,num_inserted_rows
2,2


Incremental loading so skip code script 72

In [0]:
CREATE OR REPLACE VIEW sales_scd.viewscdproducts
AS
(
SELECT DISTINCT(product_id) as product_id, product_name,product_category FROM sales_scd.orders
WHERE order_date > '2023-01-19'
)

In [0]:
SELECT * FROM sales_scd.viewscdproducts

product_id,product_name,product_category
501,Airpods,Accessories
504,Gaming laptop,Accessories


#Merge SCD TYPE 1  (UPSERT)

In [0]:
MERGE INTO sales_scd.dimproducts AS target_table
USING sales_scd.viewscdproducts AS source_table
ON target_table.product_id = source_table.product_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:

SELECT * FROM sales_scd.dimproducts

product_id,product_name,product_category
508,Tablet Screen Protector,Accessories
502,Mechanical Keyboard,Accessories
507,Bluetooth Headphones,Electronics
503,Gaming Monitor,Electronics
509,Smartwatch,Electronics
506,Smartphone X,Electronics
505,USB-C Cable,Accessories
501,Airpods,Accessories
504,Gaming laptop,Accessories
